# Tutorial: Scorey Eval DB Walkthrough

Audience:
- This is for the imagineer side of the project: someone who wants to inspect the first eval lane without having to reverse-engineer the runtime.

Prerequisites:
- Run `make install` once.
- Open this notebook from the repo root.

Learning goals:
- By the end, you can initialize the eval database.
- By the end, you can record local Scorey rounds into it.
- By the end, you can run the Beta 1.0 picks gate against stored rows.
- By the end, you can inspect counts and apply a binary judgment.


## Outline

1. Point at a safe scratch database
2. Initialize the schema
3. Record a few local rounds
4. Inspect rows and counts
5. Run the Beta 1.0 picks gate
6. Judge one row
7. Map the notebook back to the CLI surface


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from scorey.eval_db import counts, default_eval_db_path, init_db, judge_output, list_outputs, record_round_state
from scorey.eval_gates import evaluate_research_beta_1, research_beta_1_pass_pairs
from scorey.pipeline import build_local_round_state, compose_round

MAIN_DB_PATH = default_eval_db_path()
SCRATCH_DB_PATH = MAIN_DB_PATH.with_name("scorey-eval-walkthrough.sqlite")
SCRATCH_DB_PATH.relative_to(REPO_ROOT)


PosixPath('.local/scorey-eval-walkthrough.sqlite')

## Step 1 - Start with a clean scratch database

This notebook uses a scratch SQLite file so you can rerun cells without touching the main operator database at `.local/evals.sqlite`.


In [2]:
if SCRATCH_DB_PATH.exists():
    SCRATCH_DB_PATH.unlink()

init_db(SCRATCH_DB_PATH)
SCRATCH_DB_PATH.exists(), SCRATCH_DB_PATH.relative_to(REPO_ROOT)


(True,
 PosixPath('.local/scorey-eval-walkthrough.sqlite'))

## Step 2 - Record a few local rounds

The helper below uses the same runtime pieces as the CLI local mode:

- `build_local_round_state(...)`
- `compose_round(...)`
- `record_round_state(...)`


In [3]:
def record_local_round(user_pick: str, *, scorey_score: int = 1) -> int:
    round_state = build_local_round_state(user_pick, scorey_score=scorey_score)
    round_text = compose_round(round_state)
    return record_round_state(
        SCRATCH_DB_PATH,
        round_state,
        round_text,
        source_mode="local",
        model="local-fixture",
    )

recorded_ids = [
    record_local_round("rock", scorey_score=1),
    record_local_round("paper", scorey_score=2),
    record_local_round("scissors", scorey_score=3),
]
recorded_ids


[1, 2, 3]

## Step 3 - Inspect the stored rows

Each output row keeps the round text plus the small routing metadata that makes the row legible without rereading the whole runtime.


In [4]:
rows = list_outputs(SCRATCH_DB_PATH, limit=10)
[dict(row) for row in rows]


[{'id': 3,
  'user_pick': 'scissors',
  'scorey_pick': 'paper',
  'route_family': 'cross-object',
  'round_text': 'you: scissors\nme: paper\n\nmy paper beats your scissors because my paper was kitchen scissors and your scissors were a permission slip.\n\nme: 3, you: not on the board\n\nscorey.',
  'source_mode': 'local',
  'model': 'local-fixture',
  'current_verdict': 'pending',
  'current_note': '',
  'created_at': '2026-09-20T01:23:11.661743+00:00'},
 {'id': 2,
  'user_pick': 'paper',
  'scorey_pick': 'paper',
  'route_family': 'same-pick',
  'round_text': 'you: paper\nme: paper\n\nmy paper beats your paper because my paper was clipped to the answer key and your paper was damp confetti from homeroom.\n\nme: 2, you: none\n\nscorey.',
  'source_mode': 'local',
  'model': 'local-fixture',
  'current_verdict': 'pending',
  'current_note': '',
  'created_at': '2026-09-20T01:23:11.661292+00:00'},
 {'id': 1,
  'user_pick': 'rock',
  'scorey_pick': 'scissors',
  'route_family': 'cross-objec

## Step 4 - Check the verdict counts

Before judgment, every row should still be pending.


In [5]:
counts(SCRATCH_DB_PATH)


{'total': 3, 'pass': 0, 'fail': 0, 'pending': 3}

## Step 5 - Run the Beta 1.0 picks gate

Beta 1.0 only judges the pick pair in `scorey_pick, user_pick` order.

Pass pairs:
- reverse gameplay routes
- same-pick loophole routes

Fail:
- every other pair


In [6]:
row_results = []
for row in rows:
    result = evaluate_research_beta_1(row["user_pick"], row["scorey_pick"])
    row_results.append(
        {
            "scorey_pick": result.scorey_pick,
            "user_pick": result.user_pick,
            "verdict": result.verdict,
            "reason": result.reason,
        }
    )

{
    "pass_pairs": research_beta_1_pass_pairs(),
    "row_results": row_results,
}


{'pass_pairs': (('paper', 'scissors'),
  ('rock', 'paper'),
  ('scissors', 'rock'),
  ('paper', 'paper'),
  ('rock', 'rock'),
  ('scissors', 'scissors')),
 'row_results': [{'scorey_pick': 'paper',
   'user_pick': 'scissors',
   'verdict': 'pass',
   'reason': 'reverse gameplay route'},
  {'scorey_pick': 'paper',
   'user_pick': 'paper',
   'verdict': 'pass',
   'reason': 'same-pick loophole route'},
  {'scorey_pick': 'scissors',
   'user_pick': 'rock',
   'verdict': 'pass',
   'reason': 'reverse gameplay route'}]}

## Step 6 - Apply one binary judgment

Judgments are append-only history in `eval_judgments`, and the latest verdict is mirrored onto the output row for quick listing.


In [7]:
judge_output(
    SCRATCH_DB_PATH,
    recorded_ids[0],
    "pass",
    "pick-specific and easy to read",
)

counts(SCRATCH_DB_PATH)


{'total': 3, 'pass': 1, 'fail': 0, 'pending': 2}

In [8]:
[dict(row) for row in list_outputs(SCRATCH_DB_PATH, limit=10)]


[{'id': 3,
  'user_pick': 'scissors',
  'scorey_pick': 'paper',
  'route_family': 'cross-object',
  'round_text': 'you: scissors\nme: paper\n\nmy paper beats your scissors because my paper was kitchen scissors and your scissors were a permission slip.\n\nme: 3, you: not on the board\n\nscorey.',
  'source_mode': 'local',
  'model': 'local-fixture',
  'current_verdict': 'pending',
  'current_note': '',
  'created_at': '2026-09-20T01:23:11.661743+00:00'},
 {'id': 2,
  'user_pick': 'paper',
  'scorey_pick': 'paper',
  'route_family': 'same-pick',
  'round_text': 'you: paper\nme: paper\n\nmy paper beats your paper because my paper was clipped to the answer key and your paper was damp confetti from homeroom.\n\nme: 2, you: none\n\nscorey.',
  'source_mode': 'local',
  'model': 'local-fixture',
  'current_verdict': 'pending',
  'current_note': '',
  'created_at': '2026-09-20T01:23:11.661292+00:00'},
 {'id': 1,
  'user_pick': 'rock',
  'scorey_pick': 'scissors',
  'route_family': 'cross-objec

## Step 7 - Map this back to the CLI

Notebook lane:

- uses a scratch file for safety
- calls the tracked Python module functions directly

Operator lane:

- `make eval-init`
- `make eval-list EVAL_LIMIT=10`
- `make eval-beta1 EVAL_LIMIT=10`

When you are ready to use the real operator database, swap `SCRATCH_DB_PATH` for `MAIN_DB_PATH` in the setup cell above.
